In [30]:
import requests

class OpenRouter:
    def __init__(self, api_key: str, model: str = "meta-llama/llama-3.2-3b-instruct", title="openrouter-call"):
        self.api_key = api_key
        self.model = model
        self.url = "https://openrouter.ai/api/v1/chat/completions"
        self.headers = {
            "Authorization": f"Bearer {self.api_key}",
            "X-Title": title,
            "Content-Type": "application/json"
        }

    def chat(self, prompt: str, temperature=0.3, max_tokens=500):
        payload = {
            "model": self.model,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": temperature,
            "max_tokens": max_tokens
        }
        response = requests.post(self.url, headers=self.headers, json=payload)

        if response.status_code != 200:
            raise Exception(f"OpenRouter Error {response.status_code}: {response.text}")
        
        return response.json()['choices'][0]['message']['content'].strip()


In [31]:
from sentence_transformers import SentenceTransformer, util
import json

# 读取 train 数据
train_data = [json.loads(line) for line in open("generation_train.jsonl", encoding="utf-8")]
train_texts = [x["text"] for x in train_data]

# 构建编码器
encoder = SentenceTransformer("all-MiniLM-L6-v2")
train_embeddings = encoder.encode(train_texts, convert_to_tensor=True)

# 提示构造器
class SemanticPromptBuilder:
    def __init__(self, train_data, shot_num=1):
        self.train_data = train_data
        self.shot_num = shot_num

        self.schema_text = (
            "FLIGHT(FLIGHT_ID, FROM_AIRPORT, TO_AIRPORT, DEPARTURE_TIME, FLIGHT_DAYS)\n"
            "CITY(CITY_CODE, CITY_NAME)\n"
            "AIRPORT_SERVICE(CITY_CODE, AIRPORT_CODE)\n"
            "DATE_DAY(MONTH_NUMBER, DAY_NUMBER, YEAR, DAY_NAME)\n"
            "DAYS(DAYS_CODE, DAY_NAME)"
        )

    def build_prompt(self, input_question):
        query_embedding = encoder.encode(input_question, convert_to_tensor=True)
        cos_scores = util.pytorch_cos_sim(query_embedding, train_embeddings)[0]
        top_indices = cos_scores.topk(self.shot_num).indices.tolist()

        prompt = (
            "You are a SQL query generator.\n"
            "Given a natural language question and a database schema, output a valid SQL query.\n"
            "Only use the tables and columns listed below. Do not guess or hallucinate fields.\n\n"
            f"Schema:\n{self.schema_text}\n\n"
        )

        for idx in top_indices:
            sample = self.train_data[idx]
            prompt += f"# Question: {sample['text']}\nSQL: {sample['sql']}\n\n"

        prompt += f"# Question: {input_question}\nSQL:"
        return prompt

You are a SQL query generator.
Given a natural language question and a database schema, output a valid SQL query.
Only use the tables and columns listed below. Do not guess or hallucinate fields.

Schema:
FLIGHT(FLIGHT_ID, FROM_AIRPORT, TO_AIRPORT, DEPARTURE_TIME, FLIGHT_DAYS)
CITY(CITY_CODE, CITY_NAME)
AIRPORT_SERVICE(CITY_CODE, AIRPORT_CODE)
DATE_DAY(MONTH_NUMBER, DAY_NUMBER, YEAR, DAY_NAME)
DAYS(DAYS_CODE, DAY_NAME)

# Question: do you have a UA flight from BOSTON to WASHINGTON
SQL: SELECT DISTINCT FLIGHTalias0.FLIGHT_ID FROM AIRPORT_SERVICE AS AIRPORT_SERVICEalias0 , AIRPORT_SERVICE AS AIRPORT_SERVICEalias1 , CITY AS CITYalias0 , CITY AS CITYalias1 , FLIGHT AS FLIGHTalias0 WHERE ( CITYalias0.CITY_CODE = AIRPORT_SERVICEalias0.CITY_CODE AND CITYalias0.CITY_NAME = ""WASHINGTON"" AND CITYalias1.CITY_CODE = AIRPORT_SERVICEalias1.CITY_CODE AND CITYalias1.CITY_NAME = ""BOSTON"" AND FLIGHTalias0.TO_AIRPORT = AIRPORT_SERVICEalias0.AIRPORT_CODE AND FLIGHTalias0.FROM_AIRPORT = AIRPORT_SERVICE

In [27]:
builder = PromptBuilder()
prompt = builder.build_prompt("do you have a flight leaving BOSTON at 645 going to WASHINGTON")
print(prompt)


You are a SQL query generator.
Translate the given natural language question into a SQL query.
Only use the tables and columns listed below. Do not guess or hallucinate fields.
⚠️ Do not add explanations, comments, or extra questions.
⚠️ Only output a single valid SQL query.

Schema:
FLIGHT(FLIGHT_ID, FROM_AIRPORT, TO_AIRPORT, DEPARTURE_TIME)
CITY(CITY_CODE, CITY_NAME)
AIRPORT_SERVICE(CITY_CODE, AIRPORT_CODE)

# Question: do you have a flight leaving NEW YORK at 900 going to CHICAGO
SQL: SELECT DISTINCT FLIGHTalias0.FLIGHT_ID 
FROM CITY AS CITYalias0, CITY AS CITYalias1, 
     AIRPORT_SERVICE AS AIRPORT_SERVICEalias0, AIRPORT_SERVICE AS AIRPORT_SERVICEalias1, 
     FLIGHT AS FLIGHTalias0 
WHERE CITYalias0.CITY_NAME = "NEW YORK" 
  AND CITYalias1.CITY_NAME = "CHICAGO"
  AND CITYalias0.CITY_CODE = AIRPORT_SERVICEalias0.CITY_CODE
  AND CITYalias1.CITY_CODE = AIRPORT_SERVICEalias1.CITY_CODE
  AND FLIGHTalias0.FROM_AIRPORT = AIRPORT_SERVICEalias0.AIRPORT_CODE
  AND FLIGHTalias0.TO_AIRPORT =

In [28]:
import json
import re
from tqdm import tqdm
# from promptbuilder_with_schema import PromptBuilder
# from openrouter_sdk import OpenRouter

API_KEY = "YOUR_OPENROUTER_API_KEY"
MODEL = "meta-llama/llama-3.2-3b-instruct"
client = OpenRouter(api_key=API_KEY, model=MODEL)

def clean_sql(sql):
    return re.sub(r'\s+', ' ', sql.strip().lower())

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line.strip()) for line in f if line.strip()]

def evaluate(pred, refs):
    pred = clean_sql(pred)
    refs = [clean_sql(r) for r in refs]
    return pred in refs

def run_debug(train_file, test_file, schema_csv_file, shot_type="few-shot", shot_count=5, n=5):
    builder = PromptBuilder(train_file, schema_csv_file, mode=shot_type, shot_num=shot_count)
    # builder =  SemanticPromptBuilder(train_data, shot_num=1)
    test_data = load_jsonl(test_file)[:n]

    correct = 0
    log_lines = []

    for idx, item in enumerate(tqdm(test_data)):
        prompt = builder.build_prompt(item['text'])
        print(f"\n--------------------------- Prompt #{idx+1}")
        print(prompt)
        print(f"\n[Token Length]: {len(prompt.split())} words")

        try:
            gen_sql = client.chat(prompt)
        except Exception as e:
            print(f"❌ Error on item {idx}: {e}")
            continue
        gen_sql+=" ;"
        gold = item['sql']
        hit = evaluate(gen_sql, gold)
        correct += int(hit)

        print(f"🧪 Q: {item['text']}")
        print(f"🔧 SQL_PRED: {gen_sql}")
        print(f"🎯 SQL_GOLD: {gold}")
        print(f"✅ MATCH: {hit}")

        log_lines.append(json.dumps({
            "question": item['text'],
            "prompt": prompt,
            "pred": gen_sql,
            "gold": gold,
            "match": hit
        }))

    acc = correct / len(test_data)
    print(f"\n✅ [{shot_type} | {shot_count} shots] Accuracy: {acc:.4f} ({correct}/{len(test_data)})")

    with open("prompt_debug_log.jsonl", "w", encoding="utf-8") as f:
        for line in log_lines:
            f.write(line + "\n")

    print("\n📝 Log saved to prompt_debug_log.jsonl")


In [29]:
# from debug_prompt_runner import run_debug

run_debug(
    train_file="generation_train.jsonl",
    test_file="generation_test.jsonl",
    schema_csv_file="atis-schema.csv",
    shot_type="many-shot",
    shot_count=1,
    n=3  # 控制测试样本数
)


  0%|          | 0/3 [00:00<?, ?it/s]


--------------------------- Prompt #1
You are a SQL query generator.
Translate the given natural language question into a SQL query.
Only use the tables and columns listed below. Do not guess or hallucinate fields.
⚠️ Do not add explanations, comments, or extra questions.
⚠️ Only output a single valid SQL query.

Schema:
FLIGHT(FLIGHT_ID, FROM_AIRPORT, TO_AIRPORT, DEPARTURE_TIME)
CITY(CITY_CODE, CITY_NAME)
AIRPORT_SERVICE(CITY_CODE, AIRPORT_CODE)

# Question: do you have a flight leaving NEW YORK at 900 going to CHICAGO
SQL: SELECT DISTINCT FLIGHTalias0.FLIGHT_ID 
FROM CITY AS CITYalias0, CITY AS CITYalias1, 
     AIRPORT_SERVICE AS AIRPORT_SERVICEalias0, AIRPORT_SERVICE AS AIRPORT_SERVICEalias1, 
     FLIGHT AS FLIGHTalias0 
WHERE CITYalias0.CITY_NAME = "NEW YORK" 
  AND CITYalias1.CITY_NAME = "CHICAGO"
  AND CITYalias0.CITY_CODE = AIRPORT_SERVICEalias0.CITY_CODE
  AND CITYalias1.CITY_CODE = AIRPORT_SERVICEalias1.CITY_CODE
  AND FLIGHTalias0.FROM_AIRPORT = AIRPORT_SERVICEalias0.AIRPO

 33%|███▎      | 1/3 [00:03<00:06,  3.02s/it]

🧪 Q: do you have an 819 flight from DENVER to SAN FRANCISCO
🔧 SQL_PRED: SELECT DISTINCT FLIGHTalias0.FLIGHT_ID 
FROM CITY AS CITYalias0, CITY AS CITYalias1, 
     AIRPORT_SERVICE AS AIRPORT_SERVICEalias0, AIRPORT_SERVICE AS AIRPORT_SERVICEalias1, 
     FLIGHT AS FLIGHTalias0 
WHERE CITYalias0.CITY_NAME = "DENVER" 
  AND CITYalias1.CITY_NAME = "SAN FRANCISCO"
  AND CITYalias0.CITY_CODE = AIRPORT_SERVICEalias0.CITY_CODE
  AND CITYalias1.CITY_CODE = AIRPORT_SERVICEalias1.CITY_CODE
  AND FLIGHTalias0.FROM_AIRPORT = AIRPORT_SERVICEalias0.AIRPORT_CODE
  AND FLIGHTalias0.TO_AIRPORT = AIRPORT_SERVICEalias1.AIRPORT_CODE
  AND FLIGHTalias0.DEPARTURE_TIME = 819; ;
🎯 SQL_GOLD: SELECT DISTINCT FLIGHTalias0.FLIGHT_ID FROM AIRPORT_SERVICE AS AIRPORT_SERVICEalias0 , AIRPORT_SERVICE AS AIRPORT_SERVICEalias1 , CITY AS CITYalias0 , CITY AS CITYalias1 , FLIGHT AS FLIGHTalias0 WHERE ( CITYalias1.CITY_CODE = AIRPORT_SERVICEalias1.CITY_CODE AND CITYalias1.CITY_NAME = ""SAN FRANCISCO"" AND FLIGHTalias0.DEPART

 67%|██████▋   | 2/3 [00:04<00:02,  2.10s/it]

🧪 Q: i would like a flight from KANSAS CITY to ST. PAUL departing at 1600
🔧 SQL_PRED: SELECT DISTINCT FLIGHT_ID 
FROM FLIGHT 
WHERE FROM_AIRPORT = (SELECT AIRPORT_CODE FROM AIRPORT_SERVICE WHERE CITY_NAME = "KANSAS CITY") 
  AND TO_AIRPORT = (SELECT AIRPORT_CODE FROM AIRPORT_SERVICE WHERE CITY_NAME = "ST. PAUL") 
  AND DEPARTURE_TIME = 1600; ;
🎯 SQL_GOLD: SELECT DISTINCT FLIGHTalias0.FLIGHT_ID FROM AIRPORT_SERVICE AS AIRPORT_SERVICEalias0 , AIRPORT_SERVICE AS AIRPORT_SERVICEalias1 , CITY AS CITYalias0 , CITY AS CITYalias1 , FLIGHT AS FLIGHTalias0 WHERE ( CITYalias1.CITY_CODE = AIRPORT_SERVICEalias1.CITY_CODE AND CITYalias1.CITY_NAME = ""ST. PAUL"" AND FLIGHTalias0.DEPARTURE_TIME = 1600 AND FLIGHTalias0.TO_AIRPORT = AIRPORT_SERVICEalias1.AIRPORT_CODE ) AND CITYalias0.CITY_CODE = AIRPORT_SERVICEalias0.CITY_CODE AND CITYalias0.CITY_NAME = ""KANSAS CITY"" AND FLIGHTalias0.FROM_AIRPORT = AIRPORT_SERVICEalias0.AIRPORT_CODE ;
✅ MATCH: False

--------------------------- Prompt #3
You are a SQL

100%|██████████| 3/3 [00:06<00:00,  2.00s/it]

🧪 Q: do you have a flight leaving BOSTON at 645 going to WASHINGTON
🔧 SQL_PRED: SELECT DISTINCT FLIGHT_ID 
FROM CITY AS CITYalias0, CITY AS CITYalias1, 
     AIRPORT_SERVICE AS AIRPORT_SERVICEalias0, AIRPORT_SERVICE AS AIRPORT_SERVICEalias1, 
     FLIGHT AS FLIGHTalias0 
WHERE CITYalias0.CITY_NAME = "BOSTON" 
  AND CITYalias1.CITY_NAME = "WASHINGTON"
  AND CITYalias0.CITY_CODE = AIRPORT_SERVICEalias0.CITY_CODE
  AND CITYalias1.CITY_CODE = AIRPORT_SERVICEalias1.CITY_CODE
  AND FLIGHTalias0.FROM_AIRPORT = AIRPORT_SERVICEalias0.AIRPORT_CODE
  AND FLIGHTalias0.TO_AIRPORT = AIRPORT_SERVICEalias1.AIRPORT_CODE
  AND FLIGHTalias0.DEPARTURE_TIME = 645; ;
🎯 SQL_GOLD: SELECT DISTINCT FLIGHTalias0.FLIGHT_ID FROM AIRPORT_SERVICE AS AIRPORT_SERVICEalias0 , AIRPORT_SERVICE AS AIRPORT_SERVICEalias1 , CITY AS CITYalias0 , CITY AS CITYalias1 , FLIGHT AS FLIGHTalias0 WHERE ( CITYalias1.CITY_CODE = AIRPORT_SERVICEalias1.CITY_CODE AND CITYalias1.CITY_NAME = ""WASHINGTON"" AND FLIGHTalias0.DEPARTURE_TIME = 

# new all section

In [ ]:
import random
import json
import csv
import requests
from tqdm import tqdm

# ✅ 加载 JSONL 数据
def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line.strip()) for line in f]

# ✅ 从 schema CSV 中读取结构信息
def read_schema_from_csv(schema_csv_path: str) -> str:
    schema_map = {}
    with open(schema_csv_path, newline='', encoding='utf-8') as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            table = row['Table Name'].strip()
            field = row[' Field Name'].strip()
            if table == '-' or field == '-':
                continue
            if table not in schema_map:
                schema_map[table] = []
            schema_map[table].append(field)
    return "\n".join([f"{table}({', '.join(fields)})" for table, fields in schema_map.items()])

# ✅ 提示词构造器
class PromptBuilder:
    def __init__(self, train_data_path, schema_csv_path=None, mode="many-shot", shot_num=3):
        with open(train_data_path, "r", encoding="utf-8") as f:
            self.examples = [json.loads(line.strip()) for line in f]

        self.schema_text = read_schema_from_csv(schema_csv_path) if schema_csv_path else ""
        self.mode = mode
        self.shot_num = shot_num

    def build_prompt(self, input_question):
        instruction = (
            "You are a SQL query generator.\n"
            "Translate the given natural language question into a SQL query.\n"
            "Only use the schema provided.\n"
            "⚠️ Do not explain. ⚠️ Only output a single SQL query.\n\n"
        )

        prompt = instruction + f"Schema:\n{self.schema_text}\n\n"

        if self.mode != "zero-shot":
            sampled = random.sample(self.examples, min(self.shot_num, len(self.examples)))
            for ex in sampled:
                prompt += f"# Question: {ex['text']}\nSQL: {ex['sql']}\n\n"

        prompt += f"# Question: {input_question}\nSQL:"
        return prompt

# ✅ 评估函数：是否严格匹配
def evaluate(pred_sql: str, gold_sql: str) -> bool:
    return pred_sql.strip().rstrip(";") == gold_sql.strip().rstrip(";")

# ✅ OpenRouter LLM 封装类
class OpenRouter:
    def __init__(self, api_key: str, model: str = "meta-llama/llama-3.2-3b-instruct", title="openrouter-call"):
        self.api_key = api_key
        self.model = model
        self.url = "https://openrouter.ai/api/v1/chat/completions"
        self.headers = {
            "Authorization": f"Bearer {self.api_key}",
            "X-Title": title,
            "Content-Type": "application/json"
        }

    def chat(self, prompt: str, temperature=0.3, max_tokens=500):
        payload = {
            "model": self.model,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": temperature,
            "max_tokens": max_tokens
        }
        response = requests.post(self.url, headers=self.headers, json=payload)
        if response.status_code != 200:
            raise Exception(f"OpenRouter Error {response.status_code}: {response.text}")
        return response.json()['choices'][0]['message']['content'].strip()

# ✅ Debug + Accuracy 检查函数
def run_debug(train_file, test_file, schema_csv_file, shot_type="many-shot", shot_count=3, n=5, api_key=None):
    builder = PromptBuilder(train_file, schema_csv_file, mode=shot_type, shot_num=shot_count)
    test_data = load_jsonl(test_file)[:n]
    client = OpenRouter(api_key=api_key)

    correct = 0
    for idx, item in enumerate(tqdm(test_data)):
        prompt = builder.build_prompt(item["text"])
        print(f"\n--- Prompt #{idx+1} ---")
        print(prompt)

        try:
            gen_sql = client.chat(prompt)
        except Exception as e:
            print(f"❌ Error on item {idx}: {e}")
            continue

        if not gen_sql.strip().endswith(";"):
            gen_sql += " ;"

        gold = item["sql"]
        match = evaluate(gen_sql, gold)
        correct += int(match)

        print(f"🧪 Q: {item['text']}")
        print(f"🔧 SQL_PRED: {gen_sql}")
        print(f"🎯 SQL_GOLD: {gold}")
        print(f"✅ MATCH: {match}")

    acc = correct / len(test_data)
    print(f"\n✅ [{shot_type} | {shot_count} shots] Accuracy: {acc:.4f} ({correct}/{len(test_data)})")


In [35]:
run_debug(
    train_file="generation_train.jsonl",
    test_file="generation_test.jsonl",
    schema_csv_file="atis-schema.csv",
    shot_type="many-shot",
    shot_count=3,
    n=5,
    api_key="YOUR_OPENROUTER_API_KEY"
)

  0%|          | 0/5 [00:00<?, ?it/s]


--- Prompt #1 ---
You are a SQL query generator.
Translate the given natural language question into a SQL query.
Only use the schema provided.
⚠️ Do not explain. ⚠️ Only output a single SQL query.

Schema:
AIRCRAFT(AIRCRAFT_CODE, AIRCRAFT_DESCRIPTION, MANUFACTURER, BASIC_TYPE, ENGINES, PROPULSION, WIDE_BODY, WING_SPAN, LENGTH, WEIGHT, CAPACITY, PAY_LOAD, CRUISING_SPEED, RANGE_MILES, PRESSURIZED)
AIRLINE(AIRLINE_CODE, AIRLINE_NAME, NOTE)
AIRPORT(AIRPORT_CODE, AIRPORT_NAME, AIRPORT_LOCATION, STATE_CODE, COUNTRY_NAME, TIME_ZONE_CODE, MINIMUM_CONNECT_TIME)
AIRPORT_SERVICE(CITY_CODE, AIRPORT_CODE, MILES_DISTANT, DIRECTION, MINUTES_DISTANT)
CITY(CITY_CODE, CITY_NAME, STATE_CODE, COUNTRY_NAME, TIME_ZONE_CODE)
CLASS_OF_SERVICE(BOOKING_CLASS, RANK, CLASS_DESCRIPTION)
CODE_DESCRIPTION(CODE, DESCRIPTION)
COMPARTMENT_CLASS(COMPARTMENT, CLASS_TYPE)
DATE_DAY(MONTH_NUMBER, DAY_NUMBER, YEAR, DAY_NAME)
DAYS(DAYS_CODE, DAY_NAME)
DUAL_CARRIER(MAIN_AIRLINE, LOW_FLIGHT_NUMBER, HIGH_FLIGHT_NUMBER, DUAL_AIR

 20%|██        | 1/5 [00:20<01:20, 20.07s/it]

🧪 Q: do you have an 819 flight from DENVER to SAN FRANCISCO
🔧 SQL_PRED: ```sql
SELECT DISTINCT FLIGHTalias0.FLIGHT_ID FROM AIRPORT_SERVICE AS AIRPORT_SERVICEalias0 , AIRPORT_SERVICE AS AIRPORT_SERVICEalias1 , CITY AS CITYalias0 , CITY AS CITYalias1 , FLIGHT AS FLIGHTalias0 , FLIGHT_FARE AS FLIGHT_FAREalias0 , FLIGHT_LEG AS FLIGHT_LEGalias0 , FLIGHT_STOP AS FLIGHT_STOPalias0 WHERE ( CITYalias0.CITY_CODE = AIRPORT_SERVICEalias0.CITY_CODE AND CITYalias0.CITY_NAME = ""DENVER"" AND CITYalias1.CITY_CODE = AIRPORT_SERVICEalias1.CITY_CODE AND CITYalias1.CITY_NAME = ""SAN FRANCISCO"" AND FLIGHTalias0.FLIGHT_ID = FLIGHT_FAREalias0.FLIGHT_ID AND FLIGHTalias0.FLIGHT_ID = FLIGHT_LEGalias0.FLIGHT_ID AND FLIGHTalias0.FLIGHT_ID = FLIGHT_STOPalias0.FLIGHT_ID AND FLIGHTalias0.FROM_AIRPORT = AIRPORT_SERVICEalias0.AIRPORT_CODE AND FLIGHTalias0.TO_AIRPORT = AIRPORT_SERVICEalias1.AIRPORT_CODE AND FLIGHTalias0.DEPARTURE_TIME BETWEEN 1200 AND 1800 ) AND FLIGHTalias0.STOPS = 0 AND FLIGHTalias0.DUAL_CARRIER = (

 40%|████      | 2/5 [00:22<00:28,  9.61s/it]

...

🧪 Q: do you have a flight leaving BOSTON at 645 going to WASHINGTON
🔧 SQL_PRED: SELECT * FROM FLIGHT AS FLIGHTalias0 WHERE FLIGHTalias0.ARRIVAL_TIME = 645 AND FLIGHTalias0.FROM_AIRPORT = "BOSTON" AND FLIGHTalias0.AIRLINE_CODE = "WASHINGTON" ;
🎯 SQL_GOLD: SELECT DISTINCT FLIGHTalias0.FLIGHT_ID FROM AIRPORT_SERVICE AS AIRPORT_SERVICEalias0 , AIRPORT_SERVICE AS AIRPORT_SERVICEalias1 , CITY AS CITYalias0 , CITY AS CITYalias1 , FLIGHT AS FLIGHTalias0 WHERE ( CITYalias1.CITY_CODE = AIRPORT_SERVICEalias1.CITY_CODE AND CITYalias1.CITY_NAME = ""WASHINGTON"" AND FLIGHTalias0.DEPARTURE_TIME = 645 AND FLIGHTalias0.TO_AIRPORT = AIRPORT_SERVICEalias1.AIRPORT_CODE ) AND CITYalias0.CITY_CODE = AIRPORT_SERVICEalias0.CITY_CODE AND CITYalias0.CITY_NAME = ""BOSTON"" AND FLIGHTalias0.FROM_AIRPORT = AIRPORT_SERVICEalias0.AIRPORT_CODE ;
✅ MATCH: False

--- Prompt #4 ---
You are a SQL query generator.
Translate the given natural language question into a SQL query.
Only use the schema provided.
⚠️ Do not expla

 80%|████████  | 4/5 [00:25<00:04,  4.13s/it]

🧪 Q: i need a flight from PITTSBURGH to NEW YORK leaving at 1700
🔧 SQL_PRED: ```
SELECT FLIGHTalias0.FLIGHT_ID FROM AIRPORT_SERVICE AS AIRPORT_SERVICEalias0 , AIRPORT_SERVICE AS AIRPORT_SERVICEalias1 , CITY AS CITYalias0 , CITY AS CITYalias1 , FLIGHT AS FLIGHTalias0 WHERE ( CITYalias0.CITY_CODE = AIRPORT_SERVICEalias0.CITY_CODE AND CITYalias0.CITY_NAME = ""PITTSBURGH"" AND CITYalias1.CITY_CODE = AIRPORT_SERVICEalias1.CITY_CODE AND CITYalias1.CITY_NAME = ""NEW YORK"" AND FLIGHTalias0.FROM_AIRPORT = AIRPORT_SERVICEalias0.AIRPORT_CODE AND FLIGHTalias0.TO_AIRPORT = AIRPORT_SERVICEalias1.AIRPORT_CODE AND FLIGHTalias0.DEPARTURE_TIME = 1700 ) ;
``` ;
🎯 SQL_GOLD: SELECT DISTINCT FLIGHTalias0.FLIGHT_ID FROM AIRPORT_SERVICE AS AIRPORT_SERVICEalias0 , AIRPORT_SERVICE AS AIRPORT_SERVICEalias1 , CITY AS CITYalias0 , CITY AS CITYalias1 , FLIGHT AS FLIGHTalias0 WHERE ( CITYalias1.CITY_CODE = AIRPORT_SERVICEalias1.CITY_CODE AND CITYalias1.CITY_NAME = ""NEW YORK"" AND FLIGHTalias0.DEPARTURE_TIME = 1700

100%|██████████| 5/5 [00:27<00:00,  5.47s/it]

🧪 Q: is there a 400 o'clock flight from WASHINGTON to DENVER
🔧 SQL_PRED: SELECT DISTINCT FLIGHTalias0.FLIGHT_ID FROM AIRPORT_SERVICE AS AIRPORT_SERVICEalias0 , AIRPORT_SERVICE AS AIRPORT_SERVICEalias1 , CITY AS CITYalias0 , CITY AS CITYalias1 , FLIGHT AS FLIGHTalias0 WHERE CITYalias0.CITY_NAME = ""WASHINGTON"" AND CITYalias1.CITY_NAME = ""DENVER"" AND FLIGHTalias0.FLIGHT_NUMBER LIKE "%400%" AND FLIGHTalias0.FLIGHTalias0.AIRPORT_CODE = AIRPORT_SERVICEalias0.AIRPORT_CODE AND FLIGHTalias0.AIRPORT_CODE = AIRPORT_SERVICEalias1.AIRPORT_CODE ;
🎯 SQL_GOLD: SELECT DISTINCT FLIGHTalias0.FLIGHT_ID FROM AIRPORT_SERVICE AS AIRPORT_SERVICEalias0 , AIRPORT_SERVICE AS AIRPORT_SERVICEalias1 , CITY AS CITYalias0 , CITY AS CITYalias1 , FLIGHT AS FLIGHTalias0 WHERE ( CITYalias1.CITY_CODE = AIRPORT_SERVICEalias1.CITY_CODE AND CITYalias1.CITY_NAME = ""DENVER"" AND FLIGHTalias0.DEPARTURE_TIME = 400 AND FLIGHTalias0.TO_AIRPORT = AIRPORT_SERVICEalias1.AIRPORT_CODE ) AND CITYalias0.CITY_CODE = AIRPORT_SERVICEal